# 06 · Season dates from all indices and both curves, with the best choice

> **Outlook, not part of the report.** This notebook shows the next step we would take. `04`, whose results the report uses, keeps two candidates per date type and leaves PSRI and RCC out entirely. Here, instead, every index and both curves are tested against the validation data, and the version used per date type is chosen by a fixed procedure. This decides more rigorously which indices to use than discarding PSRI and RCC outright. The results of this notebook are not included in the report and are not used by `05_climate_analysis`.

Derives the start, peak and end of the larch yellowing season per year and topographic zone with the methods of `04_phenology_metrics`, but for every combination of index (NDVI, GRVI, PSRI, RCC) and curve (Whittaker smoother, double logistic): 8 candidates per date type. Each candidate is checked against the validation data, and pair averages of the best ones compete with them. The best version per date type is then used as `04` uses its chosen dates: trend and projection, topography and STL.

| Section | Content |
|---|---|
| 1 | load and filter the cached statistics; 1b: partial-swath check |
| 2 | the indices over the years |
| 3 | curves per year and zone: observations and weights (3a), Whittaker (3b), double logistic (3c), figures (3d) |
| 4 | dates per year, all candidates: start (4a), peak (4b), end (4c) |
| 5 | checks against the validation data: whole AOI (5a), the area around a local site (5b) |
| 6 | ranking: single candidates (6a), with pair averages (6b), the choice per date type (6c) |
| 7 | the chosen dates; 7b: at the local site |
| 8 | OLS trend and projection of the chosen dates |
| 9 | chosen dates vs elevation and aspect |
| 10 | STL decomposition of the indices |
| 11 | export |

- **Settings:** `config.py`, the same as `04` (date filter, weights, `WHITTAKER_LAMBDAS`, `AMPLITUDE_FRACTIONS`; `SEASON_METHODS` / `SEASON_CHOICE` for the reference row `04 choice`), plus `BEST_CHOICE` and `BEST_TOP_N` for the choice of section 6
- **Reads:**
  - `cache/<AOI>_zones/` (from `01_download_sentinel2`)
  - `cache/<AOI>_sites/` (from `01_download_sentinel2` section 6, optional)
  - `cache/validation/<AOI>/` (from `02_download_validation`)
- **Writes** to `results/<AOI>/` (separate from the files of `04`; `05_climate_analysis` reads those of `04`):
  - `season_all_indices_dates.csv`: start / peak / end of every candidate and pair average per zone and year
  - `season_all_indices_site_dates.csv`: the same for the areas around the local sites of `SITE_AREAS`
  - `season_all_indices_comparison.csv`: the checks and mean ranks of section 6
  - `season_selected_best.csv`: the chosen start / peak / end per zone and year
  - `season_best_choice.csv`: what was chosen per date type and how
  - `season_projection_best.csv`, `topography_effects_best.csv`: sections 8 and 9

Years whose autumn is not fully cached yet (`SEASON_COMPLETE_DOY`, e.g. the current year) are left out of all plots, tables, trends and models. Runs offline in about a minute. Sources are listed at the end.

## Yellowing season: start, peak and end

Each date is read off a curve at a share of the autumn amplitude still left (`AMPLITUDE_FRACTIONS`):

| Date | Meaning | Share left | `Whittaker` | `double logistic` | Checked against |
|---|---|---|---|---|---|
| **Start** | needles begin to lose chlorophyll | 90 % (MCD12Q2 "senescence") | last day before the 50 % point with ≥ 90 % left | 90 % left | needle colouring 50 %, indirect (section 5) |
| **Peak** | canopy most yellow: chlorophyll gone, needles still on | 50 % (MCD12Q2 "greendown midpoint") | first day after the summer window with ≤ 50 % left | the autumn inflection day | needle colouring 50 % |
| **End** | needles have fallen | 15 % (HR-VPP end of season, MCD12Q2 dormancy) | first day after the 50 % point with ≤ 15 % left | 15 % left | needle drop 50 % |

- **Curves:** the weighted Whittaker smoother, daily over the whole series (Eilers 2003; Atzberger & Eilers 2011), follows the observations without assuming a shape. The double logistic (Beck et al. 2006), the function TIMESAT fits for HR-VPP, assumes the shape of the seasonal curve and is fitted over the snow-free season with the base fixed.
- **Thresholds:** Gray et al. (2019) for MCD12Q2, Tian et al. (2021) for HR-VPP.
- **Expected:** start < peak < end in every year and zone; earlier at higher elevation.
- **Limits:** after mid-October there are few clear observations (low sun turns north slopes SCL "dark", clouds, early snow), so many dates, the end above all, lie in a data gap where the curve only interpolates.

Both curves are applied to each index:

| Index | Autumn change | Signal |
|---|---|---|
| NDVI | falls | mostly with needle fall: yellow needles still reflect NIR |
| GRVI | falls | green falls relative to red as chlorophyll breaks down (Motohka et al. 2010) |
| PSRI | rises | carotenoid / chlorophyll ratio during senescence (Merzlyak et al. 1999) |
| RCC | rises | red chromatic coordinate, autumn colour in PhenoCam studies (Sonnentag et al. 2012; Richardson et al. 2018) |

**Choice of the best version (section 6):**
1. The 8 single candidates are ranked per date type against the validation data.
2. The pair averages of the `BEST_TOP_N` best singles join them (with 4: 6 pairs, 14 contenders), and all are ranked again.
3. Per date type, `BEST_CHOICE` sets the date used from section 7 on: `"auto"` = the best mean rank; `"peak"` (start only) = the start of the candidate(s) that won the peak; or a fixed choice such as `"Whittaker / GRVI + double logistic / RCC"`. There is no observation of the start, so by default it comes from the same curve and index as the peak.

The reference row `04 choice` is the date `04` passes on (`SEASON_CHOICE`, from the candidates of `SEASON_METHODS`). It is ranked beside the contenders without changing their ranks, and is not chosen by `"auto"`.

**Validation:** needle colouring 50 % (`mlard94d`) and needle drop 50 % (`mlard95d`), from two separate sources: MeteoSwiss stations and the local sites of `Lärchen_pheno.xlsx`.

In [ ]:
import importlib
import warnings
from datetime import date, timedelta
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
import statsmodels.api as sm
import statsmodels.formula.api as smf

from matplotlib.colors import LightSource
from scipy import sparse
from scipy.optimize import OptimizeWarning, curve_fit
from scipy.sparse.linalg import spsolve
from scipy.stats import pearsonr
from statsmodels.tsa.seasonal import STL

import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import clean_frame, incomplete_years, index_frame, load_stats, load_zone_table

INDEX_COLORS = {"NDVI": "darkgreen", "GRVI": "darkorange", "PSRI": "purple", "RCC": "firebrick"}
INDEX_MARKERS = {"NDVI": "o", "GRVI": "s", "PSRI": "^", "RCC": "v"}
METHOD_COLORS = {"Whittaker": "tab:green", "double logistic": "tab:orange"}
ASPECT_CODES = {"N": 0, "S": 1, "flat": 2}   # aspect classes as stored in zones.tif (01)
ASPECT_COLORS = {"N": "tab:blue", "S": "tab:orange", "flat": "grey"}
METHODS = ["Whittaker", "double logistic"]
DATE_TYPES = {"start": "start", "peak": "mid", "end": "end"}   # date type -> metric of AMPLITUDE_FRACTIONS
VALIDATE_WITH = {"start": "mlard94d", "peak": "mlard94d", "end": "mlard95d"}   # observed phase per date type
CHOICE = "04 choice"
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | results: {OUT_DIR}")

## 1. Load and filter the cached statistics

A date is kept for a zone if at least `MIN_VALID_FRACTION` of the zone's pixels are clear (and it passes `MAX_SCENE_CLOUD` / `MAX_SCL_FRACTION`). With `VALID_SHARE_OF = "covered"`, this and the other SCL shares count only the zone pixels inside the scene's footprint: every second scene comes from the edge of a neighbouring orbit's swath and covers only part of the AOI, and such a scene passes if its covered part is clear and holds at least `MIN_COVERED_PIXELS` pixels. `INCOMPLETE` lists the years whose cache ends before `SEASON_COMPLETE_DOY`. Indices of `INDICES` that the cache lacks are skipped.

In [ ]:
stats_all = load_stats()
zone_table = load_zone_table()
ZONES = list(zone_table.index)   # "all" first, then elevation band x aspect
INCOMPLETE = incomplete_years(stats_all)
if INCOMPLETE:
    print(f"Autumn not fully cached, left out: {INCOMPLETE}")
    stats_all = stats_all[~stats_all.index.year.isin(INCOMPLETE)]
years = sorted(set(stats_all.index.year))

missing = [n for n in INDICES if f"{n.lower()}_median" not in stats_all or stats_all[f"{n.lower()}_median"].isna().all()]
if missing:
    print(f"Not in the cache, skipped: {missing} (re-run 01_download_sentinel2 to add them)")
INDICES = [n for n in INDICES if n not in missing]
CANDIDATES = [(method, name) for method in METHODS for name in INDICES]

frames = {name: index_frame(stats_all, name.lower()) for name in INDICES}
clean = {name: clean_frame(stats_all, name.lower()) for name in INDICES}
for name in INDICES:
    print(f"{name}: {len(clean[name])} of {len(frames[name])} dates kept")
print(f"Years: {years[0]}-{years[-1]} ({len(years)}) | zones: {len(ZONES) - 1} + whole AOI | "
      f"{len(CANDIDATES)} candidates per date type")
display(zone_table[["n_pixels", "elev_mean", "slope_mean", "northness_mean"]].round(2).join(
    pd.DataFrame({name: {z: len(clean_frame(stats_all, name.lower(), z)) for z in ZONES} for name in INDICES})
      .add_prefix("dates_")))

### 1b. Check: partial-swath scenes vs full scenes

A partial scene always sees the same strip of a zone, from a different viewing angle. If that strip is not representative, its zone median is offset from the full-scene medians. Each kept partial scene (covering < 90 % of the zone) is paired with the nearest kept full scene at most `PAIR_MAX_DAYS` days away (usually 2), both without snow.

- **`bias`:** mean difference partial − full; near 0 = no systematic offset
- **`mad`:** median absolute difference, including real change over the days in between
- **`full_mad`:** the same for pairs of full scenes ≤ 5 days apart, as the noise to compare against
- **`bias_share`:** `bias` relative to the zone's seasonal range (5th–95th percentile)

A `bias_share` of a few percent shifts a transition date by much less than a day. If it is large for a zone, set `VALID_SHARE_OF = "zone"` or raise `MIN_COVERED_PIXELS`.

In [ ]:
PAIR_MAX_DAYS = 3
PARTIAL_COVER = 0.9   # scenes covering less of the zone count as partial


def paired_diff(a, b, max_days):
    # a - b for each date of a and the nearest other date of b at most max_days away (series indexed by date)
    m = pd.merge_asof(a.rename("a").to_frame(), b.rename("b").to_frame(), left_index=True, right_index=True,
                      direction="nearest", tolerance=pd.Timedelta(days=max_days), allow_exact_matches=False)
    return (m.a - m.b).dropna()


rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)   # pandas' merge_asof tolerance on recent numpy
    for name in INDICES:
        col = f"{name.lower()}_median"
        for zone in ZONES:
            f = clean_frame(stats_all, name.lower(), zone)
            f = f[f.scl_snow_fraction <= MAX_SNOW_FRACTION]
            partial = f.covered_fraction < PARTIAL_COVER
            diff = paired_diff(f.loc[partial, col], f.loc[~partial, col], PAIR_MAX_DAYS)
            full_diff = paired_diff(f.loc[~partial, col], f.loc[~partial, col], 5)
            spread = f[col].quantile(0.95) - f[col].quantile(0.05)
            rows.append({"index": name, "zone": zone, "partial_kept": int(partial.sum()), "n_pairs": len(diff),
                         "bias": diff.mean(), "mad": diff.abs().median(), "full_mad": full_diff.abs().median(),
                         "bias_share": diff.mean() / spread})
partial_check = pd.DataFrame(rows).set_index(["index", "zone"])
print(f"VALID_SHARE_OF = {VALID_SHARE_OF!r}, MIN_COVERED_PIXELS = {MIN_COVERED_PIXELS}")
display(partial_check.round(3))

## 2. The indices over the years (whole AOI)

- **Top:** every index over the whole period: all dates (grey), the dates that pass the filter (colour) and a centred moving average (`SMOOTH_WINDOW` × 5 days) on a regular 5-day grid, with gaps of up to 30 days interpolated.
- **Bottom:** the moving average of each year over the snow-free season (`SEASON_WINDOW`), one line per year.

The moving average is only for display; the date methods work on the observations directly.

In [ ]:
def smooth(series):
    # Centred moving average on a regular 5-day grid; gaps of up to 30 days interpolated, longer ones left empty
    regular = series.resample("5D").mean().interpolate(method="time", limit=6)
    rolled = regular.rolling(SMOOTH_WINDOW, center=True, min_periods=3).mean()
    return rolled.where(regular.notna())


def clear_limits(name, margin=0.15):
    # y limits from the clear dates (snow and cloud values of the unfiltered dates would squash the plot)
    lo, hi = medians[name].quantile([0.005, 0.995])
    return lo - margin * (hi - lo), hi + margin * (hi - lo)


medians = {name: clean[name][f"{name.lower()}_median"] for name in INDICES}
smoothed = {name: smooth(s) for name, s in medians.items()}

fig, axes = plt.subplots(len(INDICES), 1, figsize=(13, 2.4 * len(INDICES)), sharex=True, squeeze=False)
for ax, name in zip(axes[:, 0], INDICES):
    ax.scatter(frames[name].index, frames[name][f"{name.lower()}_median"], s=5, color="lightgrey", label="all dates")
    ax.scatter(medians[name].index, medians[name], s=8, color=INDEX_COLORS[name], label="clear dates")
    ax.plot(smoothed[name].index, smoothed[name], color="black", lw=1.2,
            label=f"{SMOOTH_WINDOW * 5}-day moving average")
    ax.set_ylim(*clear_limits(name))
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
axes[0, 0].legend(loc="lower left", ncol=3, fontsize=8)
fig.suptitle("Indices over the years (whole AOI)")
plt.tight_layout()
plt.show()

year_colors = dict(zip(years, plt.cm.viridis(np.linspace(0, 1, len(years)))))
fig, axes = plt.subplots(1, len(INDICES), figsize=(4.5 * len(INDICES), 3.8), squeeze=False)
for ax, name in zip(axes[0], INDICES):
    s = smoothed[name]
    for year, color in year_colors.items():
        y = s[s.index.year == year]
        ax.plot(y.index.dayofyear, y, color=color, lw=1.2, label=str(year))
    ax.set_xlim(*SEASON_WINDOW)
    ax.set_ylim(*clear_limits(name))
    ax.set_title(name)
    ax.set_xlabel("day of year")
    ax.grid(alpha=0.3)
axes[0, -1].legend(fontsize=7, ncol=2)
fig.suptitle(f"Seasonal course per year ({SMOOTH_WINDOW * 5}-day moving average, whole AOI)")
plt.tight_layout()
plt.show()

## 3. Curves fitted per year and zone

Both curves are fitted to the clear observations of every index, zone and year. Overview of the methods: Zeng et al. 2020.

### 3a. Observations, weights, outliers and the post-season level

- **Weights** follow the HR-VPP quality classes: 1 for a good date (clear share ≥ `GOOD_VALID_FRACTION`), 0.5 for a usable one, 0 for a date with snow on more than `MAX_SNOW_FRACTION` of the zone.
- **Outliers** follow the HR-VPP rule: a date is dropped (weight 0) if its residual from a stiff smoothing curve exceeds `OUTLIER_FACTOR` × the global and × the local (91-day) median residual.
- **Post-season level ("base"):** the snow-free level after needle fall, the median of the clear dates in `POST_WINDOW` over all years. It is held fixed in the fits instead of being fitted per year, as Beck et al. (2006) do with the winter NDVI. Zones with fewer than `MIN_BASE_OBS` such dates use the whole AOI's level. For PSRI and RCC this is the high level, as they rise during the autumn.

In [ ]:
def obs_frame(name, zone, stats=None, zones=None):
    # Clear observations of one index and zone: y (zone median), w (weight), year, doy. Weights as the HR-VPP
    # quality classes (ATBD; Tian et al. 2021): 1 = good, 0.5 = usable, 0 = snow (kept for the plots, not fitted).
    # stats / zones: statistics and zone table of another cache (the site areas of 5b); default: the zones
    f = clean_frame(stats_all if stats is None else stats, name.lower(), zone, zones)
    obs = pd.DataFrame({"y": f[f"{name.lower()}_median"],
                        "w": np.where(f.valid_fraction >= GOOD_VALID_FRACTION, 1.0, 0.5)}, index=f.index)
    obs.loc[f.scl_snow_fraction > MAX_SNOW_FRACTION, "w"] = 0.0
    obs = obs.dropna(subset=["y"])
    return obs.assign(year=obs.index.year, doy=obs.index.dayofyear, outlier=False)


def base_levels(name):
    # Post-season level per zone, held fixed in all fits (Beck et al. 2006). Fallbacks: the whole AOI's level; then
    # the 5th percentile of all clear dates (95th for rising indices)
    rows = {}
    for zone in ZONES:
        o = obs_by_zone[name][zone]
        post = o[o.doy.between(*POST_WINDOW) & (o.w > 0)]
        rows[zone] = {"level": post.y.median(), "n_dates": len(post), "source": "zone"}
    for zone, r in rows.items():
        if r["n_dates"] < MIN_BASE_OBS and rows["all"]["n_dates"] >= MIN_BASE_OBS:
            r.update(level=rows["all"]["level"], source="whole AOI")
        elif r["n_dates"] < MIN_BASE_OBS:
            q = 0.05 if EXPECT_DECREASE[name] else 0.95
            r.update(level=obs_by_zone[name][zone].y.quantile(q), source=f"{q:.0%} percentile")
    return pd.DataFrame(rows).T.astype({"level": float, "n_dates": int})


def whittaker(days, y, w, lam, n):
    # Eilers (2003), for vegetation indices Atzberger & Eilers (2011): on a daily grid of n days, minimise
    # sum w (y - z)^2 + lam * sum (2nd difference of z)^2. Days without an observation have weight 0
    W, Y = np.zeros(n), np.zeros(n)
    np.add.at(W, days, w)
    np.add.at(Y, days, w * y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n - 2, n))
    return spsolve((sparse.diags(W) + lam * (D.T @ D)).tocsc(), Y)


def smooth_series(obs, lam, base):
    # Daily Whittaker curve over whole calendar years. Off-season gaps of more than 90 days without a good date
    # get control points at the base level (weight 0.5) every 15 days, as in HR-VPP
    o = obs[obs.w > 0]
    start = pd.Timestamp(obs.index.min().year, 1, 1)
    end = pd.Timestamp(obs.index.max().year, 12, 31)
    grid = pd.date_range(start, end)
    days, y, w = (o.index - start).days.values, o.y.values, o.w.values
    control = [d for a, b in zip(days[:-1], days[1:]) if b - a > 90 for d in range(a + 15, b - 7, 15)
               if not 80 < grid[d].dayofyear < 266]
    days = np.concatenate([days, control]).astype(int)
    y = np.concatenate([y, np.full(len(control), base)])
    w = np.concatenate([w, np.full(len(control), 0.5)])
    return pd.Series(whittaker(days, y, w, lam, len(grid)), index=grid)


def flag_outliers(obs, base):
    # HR-VPP outlier rule (ATBD 4.2.1): residuals from a stiff curve; outlier if the residual exceeds
    # OUTLIER_FACTOR x the global median residual and OUTLIER_FACTOR x the median of the surrounding 91 days
    o = obs[obs.w > 0]
    curve = smooth_series(obs, max(WHITTAKER_LAMBDAS), base)
    res = (o.y - curve.reindex(o.index).values).abs()
    local = res.rolling("91D", center=True, min_periods=1).median()
    out = res.index[(res > OUTLIER_FACTOR * res.median()) & (res > OUTLIER_FACTOR * local)]
    obs = obs.copy()
    obs.loc[out, ["w", "outlier"]] = [0.0, True]
    return obs


obs_by_zone = {name: {zone: obs_frame(name, zone) for zone in ZONES} for name in INDICES}
base = {name: base_levels(name) for name in INDICES}
for name in INDICES:
    for zone in ZONES:
        obs_by_zone[name][zone] = flag_outliers(obs_by_zone[name][zone], base[name].level[zone])

display(pd.concat({name: b for name, b in base.items()}, axis=1).round(3))
print("Dates per index and zone: weight 1 / 0.5 / 0 (snow) / outliers")
display(pd.DataFrame({name: {zone: f"{(o.w == 1).sum()} / {(o.w == 0.5).sum()} / {((o.w == 0) & ~o.outlier).sum()} / "
                                   f"{o.outlier.sum()}" for zone, o in by_zone.items()}
                      for name, by_zone in obs_by_zone.items()}))

### 3b. Whittaker smoother and its dates

λ is chosen per index from `WHITTAKER_LAMBDAS` by 10-fold cross-validation on the whole AOI (weighted RMSE of left-out dates), then used for every zone. HR-VPP assumes clear Sentinel-2 dates are unbiased and applies no upper-envelope correction; neither does this notebook.

**Dates.** Per year, the summer level `hi` is the curve's median in `SUMMER_WINDOW`; the base is the post-season level (3a). With `r` = share of the amplitude left:
- mid (the peak) = first day after the summer window with `r ≤ 0.5`
- start = last day before mid with `r ≥ 0.9` (MCD12Q2: "last date above 90 %")
- end = first day after mid with `r ≤ 0.15`

A date that is not reached within the year stays empty, as does a year whose summer level lies on the wrong side of the base.

In [ ]:
def choose_lambda(obs, base_level, k=10, seed=0):
    # Weighted RMSE of left-out good dates for every lam (k-fold cross-validation)
    good = obs[obs.w > 0]
    fold = np.random.default_rng(seed).integers(0, k, len(good))
    rmse = {}
    for lam in WHITTAKER_LAMBDAS:
        sq = []
        for i in range(k):
            test = good[fold == i]
            curve = smooth_series(obs.drop(index=test.index), lam, base_level)
            sq.append(test.w * (test.y - curve.reindex(test.index).values) ** 2)
        rmse[lam] = np.sqrt(pd.concat(sq).sum() / good.w.sum())
    return pd.Series(rmse)


def whittaker_dates(curve, hi, lo):
    # Start / mid / end on one year of a daily curve (index = DOY)
    r = (curve - lo) / (hi - lo)
    below_mid = r[(r.index > SUMMER_WINDOW[1]) & (r <= AMPLITUDE_FRACTIONS["mid"])]
    if below_mid.empty:
        return None
    mid = below_mid.index[0]
    above = r[(r.index < mid) & (r >= AMPLITUDE_FRACTIONS["start"])]
    below = r[(r.index > mid) & (r <= AMPLITUDE_FRACTIONS["end"])]
    return {"start": above.index[-1] if not above.empty else np.nan, "mid": mid,
            "end": below.index[0] if not below.empty else np.nan}


def zone_whittaker_dates(name, zone, obs, lo, curve_all):
    # Whittaker dates of one index and zone per year (rows of the date table)
    rows = []
    for year in sorted(obs.year.unique()):
        curve = curve_all[curve_all.index.year == year]
        curve.index = curve.index.dayofyear
        n_obs = int(((obs.year == year) & (obs.w > 0)).sum())
        hi = curve.loc[SUMMER_WINDOW[0]:SUMMER_WINDOW[1]].median()
        days = whittaker_dates(curve, hi, lo) if (hi > lo) == EXPECT_DECREASE[name] else None
        if days:
            rows += [{"index": name, "zone": zone, "year": year, "method": "Whittaker", "metric": m,
                      "doy": d, "n_obs": n_obs} for m, d in days.items()]
    return rows


cv_lambda = pd.DataFrame({name: choose_lambda(obs_by_zone[name]["all"], base[name].level["all"]) for name in INDICES})
LAMBDA = cv_lambda.idxmin()
print("Cross-validated RMSE per lambda (whole AOI); chosen:", LAMBDA.to_dict())
display(cv_lambda.rename_axis("lambda").round(4))

wcurves = {name: {zone: smooth_series(obs_by_zone[name][zone], LAMBDA[name], base[name].level[zone]) for zone in ZONES}
           for name in INDICES}
whittaker_rows = [row for name in INDICES for zone in ZONES
                  for row in zone_whittaker_dates(name, zone, obs_by_zone[name][zone], base[name].level[zone],
                                                  wcurves[name][zone])]
print(f"{len(whittaker_rows)} Whittaker dates")

### 3c. Double logistic and its dates

- **Fit:** the function of Beck et al. (2006) over the whole snow-free season (`SEASON_WINDOW`), `y = b + (m − b)·(1/(1 + exp(−mS(t − S))) + 1/(1 + exp(mA(t − A))) − 1)`, with the base `b` fixed at the post-season level (3a) and the dates weighted. It is the function type TIMESAT fits for HR-VPP; only its autumn half (`A`, `mA`) gives dates here. For PSRI and RCC the summer level `m` lies below `b`, and the same function describes the rise. A year needs at least `MIN_OBS_PER_YEAR` weighted dates between `AUTUMN_START_DOY` and the end of `POST_WINDOW`.
- **Dates:** a share `f` of the amplitude is left at `t = A + ln((1 − f) / f) / mA`; the 50 % point is `A`.

`all_dates` holds the dates of both curves for every index, zone and year (long format: `method`, `index`, `metric` = start / mid / end, `doy`, `date`, `n_obs` = clear dates of the year used for the curve).

In [ ]:
def double_logistic(t, m, S, mS, A, mA, b):
    # Beck et al. (2006): b = off-season level (fixed), m = summer level, S / A = spring / autumn inflection day,
    # mS / mA = their rates (1/day)
    return b + (m - b) * (1 / (1 + np.exp(-mS * (t - S))) + 1 / (1 + np.exp(mA * (t - A))) - 1)


def fraction_day(t0, s, f):
    # Day on a logistic when a share f of the amplitude is left
    return t0 + s * np.log((1 - f) / f)


def fit_double_logistic(doy, y, w, b):
    summer = y[(doy >= SUMMER_WINDOW[0]) & (doy <= SUMMER_WINDOW[1])]
    p0 = [np.clip(np.median(summer if len(summer) else y), -1.9, 1.9), 160, 0.1, 280, 0.1]
    bounds = ([-2, SEASON_WINDOW[0], 1 / 40, SUMMER_WINDOW[0], 1 / 40], [2, SUMMER_WINDOW[0], 1, SEASON_WINDOW[1], 1])
    popt, _ = curve_fit(lambda t, m, S, mS, A, mA: double_logistic(t, m, S, mS, A, mA, b), doy, y, p0=p0,
                        sigma=1 / np.sqrt(w), bounds=bounds, maxfev=20000)
    return popt


def zone_dl_dates(name, zone, obs, lo):
    # Double-logistic fits of one index and zone per year: rows of the date table, {(index, zone, year): parameters}
    rows, fits = [], {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", OptimizeWarning)
        for year, o in obs[obs.w > 0].groupby("year"):
            autumn = o[o.doy.between(AUTUMN_START_DOY, POST_WINDOW[1])]
            if len(autumn) < MIN_OBS_PER_YEAR:
                continue
            season = o[o.doy.between(*SEASON_WINDOW)]
            try:
                m, S, mS, A, mA = fit_double_logistic(season.doy.values.astype(float), season.y.values,
                                                      season.w.values, lo)
            except RuntimeError:
                continue
            fits[(name, zone, year)] = (m, S, mS, A, mA, lo)
            rows += [{"index": name, "zone": zone, "year": year, "method": "double logistic", "metric": metric,
                      "doy": fraction_day(A, 1 / mA, f), "n_obs": len(season)}
                     for metric, f in AMPLITUDE_FRACTIONS.items()]
    return rows, fits


dl_rows, dl_fits = [], {}
for name in INDICES:
    for zone in ZONES:
        rows, fits = zone_dl_dates(name, zone, obs_by_zone[name][zone], base[name].level[zone])
        dl_rows += rows
        dl_fits.update(fits)
print(f"{len(dl_fits)} double-logistic fits")

KEYS = ["method", "index", "metric"]
all_dates = pd.DataFrame(whittaker_rows + dl_rows).dropna(subset=["doy"])
all_dates["date"] = pd.to_datetime(all_dates.year.astype(str)) + pd.to_timedelta(all_dates.doy.round() - 1, unit="D")
all_dates = all_dates.sort_values([*KEYS, "zone", "year"]).reset_index(drop=True)
print(f"{len(all_dates)} dates")

### 3d. The curves of the whole AOI with their dates

One figure per index, one column per year: the clear dates (× = weight 0: snow or outlier), the Whittaker (green) and double-logistic (orange) curve with their start (dashed), peak (solid) and end (dash-dotted), and the post-season level (grey dotted). The observed dates of that year are marked at the top: needle colouring 50 % filled, needle drop 50 % open; MeteoSwiss stations as black squares (they lie lower than the AOI), local sites as purple diamonds.

In [ ]:
REFERENCES = {"ms": ("meteoswiss_larch.csv", "MeteoSwiss", "stations", dict(color="black", ls="--", marker="s")),
              "loc": ("local_larch.csv", "local", "sites", dict(color="#7b3294", ls=":", marker="D"))}
AOI_ELEV = zone_table.elev_mean["all"]

# ---- validation data per source (a source is skipped if its file is missing)
pheno = {}
for key, (file, source, kind, _) in REFERENCES.items():
    if not (VALIDATION_DIR / file).exists():
        print(f"No {source} observations (run 02_download_validation)")
        continue
    p = pd.read_csv(VALIDATION_DIR / file)
    p = p[p.year.isin(years) & p.param.isin(list(PHENO_PARAMS))]
    p["anomaly"] = p.doy - p.groupby(["station", "param"]).doy.transform("mean")   # each station vs its own mean
    pheno[key] = p.groupby(["param", "year"]).agg(anomaly=("anomaly", "mean"), doy=("doy", "mean"),
                                                  n_stations=("station", "nunique"), height=("height_masl", "mean"))
    print(f"{source}: {p.station.nunique()} {kind}, mean height {p.height_masl.mean():.0f} m; observations: "
          + ", ".join(f"{PHENO_PARAMS[k]} {n}" for k, n in p.param.value_counts().items()))


def reference(key, target):
    # Observed dates of one source for a date type: DataFrame (year -> anomaly, doy, n_stations, height)
    return pheno[key].xs(VALIDATE_WITH[target], level="param")


CURVE_DOY = np.arange(SEASON_WINDOW[0], SEASON_WINDOW[1] + 1)
DATE_STYLES = {"start": "--", "mid": "-", "end": "-."}
date_lookup = all_dates.set_index(["method", "index", "metric", "zone", "year"]).doy


def curve_of(method, name, zone, year):
    if method == "Whittaker":
        z = wcurves[name][zone]
        z = z[z.index.year == year]
        return pd.Series(z.values, index=z.index.dayofyear).loc[SEASON_WINDOW[0]:SEASON_WINDOW[1]]
    p = dl_fits.get((name, zone, year))
    return None if p is None else pd.Series(double_logistic(CURVE_DOY, *p), index=CURVE_DOY)


for name in INDICES:
    fig, axes = plt.subplots(1, len(years), figsize=(2.1 * len(years), 3.1), sharey=True, squeeze=False)
    o_all = obs_by_zone[name]["all"]
    for ax, year in zip(axes[0], years):
        o = o_all[o_all.year == year]
        ax.scatter(o[o.w > 0].doy, o[o.w > 0].y, s=6, color=INDEX_COLORS[name])
        ax.scatter(o[o.w == 0].doy, o[o.w == 0].y, s=12, marker="x", color=INDEX_COLORS[name])
        for method in METHODS:
            z = curve_of(method, name, "all", year)
            if z is not None:
                ax.plot(z.index, z.values, color=METHOD_COLORS[method], lw=1.2)
            for metric, ls in DATE_STYLES.items():
                if (method, name, metric, "all", year) in date_lookup.index:
                    ax.axvline(date_lookup[(method, name, metric, "all", year)], color=METHOD_COLORS[method], ls=ls,
                               lw=1)
        for key, (_, _, _, style) in REFERENCES.items():
            for param, filled in [("mlard94d", True), ("mlard95d", False)]:
                if key in pheno and (param, year) in pheno[key].index:
                    ax.plot(pheno[key].doy[(param, year)], 0.95 if key == "ms" else 0.87, style["marker"], ms=5,
                            color=style["color"], mfc=style["color"] if filled else "white",
                            transform=ax.get_xaxis_transform())
        ax.axhline(base[name].level["all"], color="grey", ls=":", lw=0.8)
        ax.set_xlim(AUTUMN_START_DOY, SEASON_WINDOW[1])
        ax.set_title(str(year), fontsize=9)
        ax.set_xlabel("DOY", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel(name)
    axes[0, 0].set_ylim(*clear_limits(name))
    handles = [plt.Line2D([], [], color=METHOD_COLORS[m], lw=1.5, label=f"{m} curve") for m in METHODS]
    handles += [plt.Line2D([], [], color="black", ls=ls, lw=1, label={"mid": "peak"}.get(m, m))
                for m, ls in DATE_STYLES.items()]
    handles += [plt.Line2D([], [], color=REFERENCES[k][3]["color"], marker=REFERENCES[k][3]["marker"], ls="",
                           mfc=REFERENCES[k][3]["color"] if filled else "white",
                           label=f"{REFERENCES[k][1]} {PHENO_PARAMS[param]}")
                for k in pheno for param, filled in [("mlard94d", True), ("mlard95d", False)]]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=8)
    fig.suptitle(f"{name}: curves and dates per year (whole AOI, Jul-Nov)")
    plt.tight_layout(rect=(0, 0.14, 1, 1))
    plt.show()

## 4. Dates per year, all candidates

`series[date type][(method, index)]` holds the date of a candidate per zone and year. An average of several candidates (`combine`, as `"average"` in `04`) takes each one's anomaly from its own mean over all zone-years, averages the anomalies of the candidates present and adds the mean level of all of them, so a candidate missing in one year does not shift that year's level. `04 choice` is the date `04` passes on (`SEASON_CHOICE` from the candidates of `SEASON_METHODS`).

Per date type:
- **Table:** whole-AOI day of year per candidate and year.
- **Figure:** whole-AOI dates, one panel per curve, one line per index, the `04 choice` in grey, and the observed dates of both validation sources: needle colouring 50 % for the start and the peak, needle drop 50 % for the end. The start comes before the observed colouring by definition; it is shown for its year-to-year course.

In [ ]:
def candidate_series(dates, method, name, metric, unit="zone"):
    g = dates[(dates.method == method) & (dates["index"] == name) & (dates.metric == metric)]
    return g.set_index([unit, "year"]).doy.sort_index()


def combine(series_list):
    # Mean of several candidates per zone-year: each one's anomaly from its own mean, plus the mean level of all
    s = pd.concat(series_list, axis=1, ignore_index=True)
    return (s.sub(s.mean()).mean(axis=1) + s.mean().mean()).rename("doy")


def label(comps):
    # (("Whittaker", "GRVI"), ("double logistic", "RCC")) -> "Whittaker / GRVI + double logistic / RCC"
    return " + ".join(" / ".join(c) for c in comps)


def parse(text):
    # Inverse of label; checks that every part is a candidate
    comps = tuple(tuple(p.strip() for p in part.split("/")) for part in text.split("+"))
    bad = [c for c in comps if c not in CANDIDATES]
    if bad:
        raise ValueError(f"{text!r}: {bad} not among the candidates {[' / '.join(c) for c in CANDIDATES]}")
    return comps


def choice_04(dates, target, unit="zone"):
    # 04's chosen date of one date type (SEASON_CHOICE from the candidates of SEASON_METHODS)
    choice = SEASON_CHOICE[target]
    keys = [tuple(p.strip() for p in c.split("/")) for c in (SEASON_METHODS[target] if choice == "average" else [choice])]
    return combine([candidate_series(dates, m, n, metric, unit) for m, n, metric in keys])


def candidate_dates(dates, unit="zone"):
    # {date type: {(method, index): series}} of the 8 single candidates
    return {t: {c: candidate_series(dates, *c, metric, unit) for c in CANDIDATES} for t, metric in DATE_TYPES.items()}


for target in DATE_TYPES:
    print(f"{CHOICE}, {target}: {SEASON_CHOICE[target]} of {'; '.join(SEASON_METHODS[target])}")
series = candidate_dates(all_dates)
series_04 = {t: choice_04(all_dates, t) for t in DATE_TYPES}


def aoi(s):
    return s.xs("all", level="zone") if "all" in s.index else pd.Series(dtype=float)


aoi_dates = {t: pd.DataFrame({**{label((c,)): aoi(s) for c, s in series[t].items()}, CHOICE: aoi(series_04[t])})
             for t in DATE_TYPES}


def show_dates(target):
    display(aoi_dates[target].T.round(0))
    fig, axes = plt.subplots(1, len(METHODS), figsize=(13, 4.2), sharey=True, squeeze=False)
    for ax, method in zip(axes[0], METHODS):
        for name in INDICES:
            s = aoi_dates[target][f"{method} / {name}"]
            ax.plot(s.index, s.values, color=INDEX_COLORS[name], marker=INDEX_MARKERS[name], lw=1, ms=6, label=name)
        ax.plot(aoi_dates[target].index, aoi_dates[target][CHOICE], color="grey", lw=3, alpha=0.5, label=CHOICE)
        for key in pheno:
            _, source, kind, style = REFERENCES[key]
            r = reference(key, target)
            ax.plot(r.index, r.doy, lw=1.5, ms=5, label=f"{source} {PHENO_PARAMS[VALIDATE_WITH[target]]} "
                                                        f"({kind}, mean {r.height.mean():.0f} m)", **style)
        ax.set_title(f"{method}: {target} (whole AOI, mean {AOI_ELEV:.0f} m)")
        ax.set_xlabel("year")
        ax.xaxis.get_major_locator().set_params(integer=True)
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel("day of year")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=8)
    plt.tight_layout(rect=(0, 0.13, 1, 1))
    plt.show()

### 4a. Start of the yellowing (90 % of the amplitude left)

In [ ]:
show_dates("start")

### 4b. Peak of the yellowing (50 % left)

In [ ]:
show_dates("peak")

### 4c. End of the season (15 % left)

In [ ]:
show_dates("end")

## 5. Checks against the validation data

### 5a. Whole AOI vs MeteoSwiss stations and local sites

Per date type and candidate, over the complete years:

| Column | Meaning |
|---|---|
| `coverage` | share of zone-years with a date |
| `order_ok` | share of zone-years in the order start < peak < end, judged against the median of the 8 candidates of the neighbouring date types |
| `days_per_100m` | within-year elevation gradient across zones; negative = earlier higher up (Migliavacca et al. 2008) |
| `aoi_mean_doy`, `aoi_sd_years` | mean and year-to-year standard deviation of the whole-AOI date |
| `ms_r`, `ms_p`, `ms_rmse`, `ms_n` | whole-AOI anomalies (each series minus its own mean) vs the mean anomaly of the MeteoSwiss stations: correlation, its p-value, RMSE in days, number of years |
| `ms_offset` | mean AOI date − mean station date. The stations lie lower than the AOI, so this is not an error |
| `loc_*` | the same against the local sites (`local_larch.csv`). They lie at about the AOI's elevation, so `loc_offset` checks the level for the peak and the end |

Observed phase per date type: needle colouring 50 % for the peak, needle drop 50 % for the end. There is no observation of the start of the yellowing; it is compared with needle colouring 50 % as an indirect check: a start that moves with the colouring from year to year gives a high `r`, but its offset is expected and says nothing. With 9 years, a correlation needs r > 0.67 to be significant at 5 %.

In [ ]:
consensus = pd.DataFrame({t: pd.concat(list(by_c.values()), axis=1).median(axis=1) for t, by_c in series.items()})
n_zone_years = (len(ZONES) - 1) * len(years)
AOI_COLUMNS = ["coverage", "order_ok", "days_per_100m", "aoi_mean_doy", "aoi_sd_years"]


def elevation_slope(s):
    # Days per 100 m from zone dates, within years (each year's mean removed from date and elevation)
    z = s.drop(index="all", level="zone", errors="ignore").reset_index()
    z["x"] = z.zone.map(zone_table.elev_mean) / 100
    z = z[z.groupby("year").x.transform("size") >= 2]
    x, y = z.x - z.groupby("year").x.transform("mean"), z.doy - z.groupby("year").doy.transform("mean")
    return (x @ y) / (x @ x) if len(z) > 3 and (x @ x) > 0 else np.nan


def order_share(target, s):
    c = consensus.reindex(s.index)
    ok = {"start": s < c.peak, "peak": (c.start < s) & (s < c.end), "end": s > c.peak}[target]
    known = {"start": c.peak.notna(), "peak": c.start.notna() & c.end.notna(), "end": c.peak.notna()}[target]
    return ok[known].mean() if known.any() else np.nan


def agreement(est, obs_doy, obs_anomaly=None):
    # r, p, rmse of the anomalies (each series minus its own mean over the common years), n, offset, mae
    common = est.dropna().index.intersection(obs_doy.dropna().index)
    out = {"n": len(common)}
    if len(common) < 4:
        return out
    seen = (obs_doy if obs_anomaly is None else obs_anomaly)[common]
    a, s = est[common] - est[common].mean(), seen - seen.mean()
    r, p = pearsonr(a, s)
    out.update(r=r, p=p, rmse=np.sqrt(((a - s) ** 2).mean()), offset=(est[common] - obs_doy[common]).mean(),
               mae=(est[common] - obs_doy[common]).abs().mean())
    return out


def aoi_checks(target, s):
    # Checks of one date series (zone, year) of a date type: plausibility and whole AOI vs the validation sources
    zones_only = s.drop(index="all", level="zone", errors="ignore").dropna()
    a = aoi(s)
    row = {"coverage": len(zones_only) / n_zone_years, "order_ok": order_share(target, zones_only),
           "days_per_100m": elevation_slope(s), "aoi_mean_doy": a.mean(), "aoi_sd_years": a.std()}
    for key in pheno:
        r = reference(key, target)
        row.update({f"{key}_{k}": v for k, v in agreement(a, r.doy, r.anomaly).items() if k != "mae"})
    return row


single_checks = pd.DataFrame([{"target": t, "candidate": label((c,)), **aoi_checks(t, s)}
                              for t, by_c in series.items() for c, s in by_c.items()]
                             + [{"target": t, "candidate": CHOICE, **aoi_checks(t, s)} for t, s in series_04.items()]
                             ).set_index(["target", "candidate"])
for target in DATE_TYPES:
    print(f"{target} vs {PHENO_PARAMS[VALIDATE_WITH[target]]}" + (" (indirect)" if target == "start" else ""))
    display(single_checks.xs(target, level="target").round(2))

### 5b. Check at a local site

The AOI pixels within `SITE_RADIUS_M` of each site in `SITE_AREAS` (statistics from `01` section 6) get their own dates, computed as for the zones (date filter, weights, outliers, λ of 3b, both curves). Post-season level: the area's own, or the whole AOI's with fewer than `MIN_BASE_OBS` dates. They are compared, year by year, with the site's own observations (phases as in 5a): `r`, `p`, `rmse` of the anomalies, `n` years, `offset` = mean estimated − observed date, `mae` = mean absolute difference of the dates. Area and site are in the same place, so for the peak and the end `offset` and `mae` are errors, apart from the elevation difference between the area and the site. With 8 years, `r` needs > 0.71 to be significant at 5 %.

**Figure:** per site, the estimated dates per year (one row per date type, one column per curve) and the observed dates (purple).

In [ ]:
SITES, site_names, site_series, site_series_04 = [], {}, {}, {}
observed_file = VALIDATION_DIR / "local_larch.csv"
if not SITE_AREAS:
    print("No SITE_AREAS in config.py")
elif not (SITE_CACHE_DIR / "zones.csv").exists() or not observed_file.exists():
    print("No site statistics or local observations (run 01_download_sentinel2 section 6 and 02_download_validation)")
else:
    site_stats = load_stats(SITE_CACHE_DIR)
    site_zones = load_zone_table(SITE_CACHE_DIR)
    SITES = list(site_zones.index)
    site_years = [y for y in years if y not in incomplete_years(site_stats)]
    observed = pd.read_csv(observed_file)
    observed = observed[observed.year.isin(site_years)]
    site_names = pd.read_csv(VALIDATION_DIR / "local_sites.csv", index_col="station").name.to_dict()

    rows = []
    for site in SITES:
        for name in INDICES:
            obs = obs_frame(name, site, site_stats, site_zones)
            post = obs[obs.doy.between(*POST_WINDOW) & (obs.w > 0)]
            lo = post.y.median() if len(post) >= MIN_BASE_OBS else base[name].level["all"]
            obs = flag_outliers(obs, lo)
            rows += zone_whittaker_dates(name, site, obs, lo, smooth_series(obs, LAMBDA[name], lo))
            rows += zone_dl_dates(name, site, obs, lo)[0]
    site_dates = pd.DataFrame(rows).rename(columns={"zone": "site"}).dropna(subset=["doy"])
    site_dates = site_dates[site_dates.year.isin(site_years)].reset_index(drop=True)
    site_series = candidate_dates(site_dates, unit="site")
    site_series_04 = {t: choice_04(site_dates, t, unit="site") for t in DATE_TYPES}
    display(site_zones.join(pd.Series(site_names, name="name")).round(1))


def at_site(s, site):
    return s.xs(site, level="site") if site in s.index.get_level_values("site") else pd.Series(dtype=float)


def seen_at(site, target):
    o = observed[(observed.station == site) & (observed.param == VALIDATE_WITH[target])]
    return o.set_index("year").doy.sort_index()


def site_checks(target, s):
    # Columns <site>_r, _p, _rmse, _n, _offset, _mae of one date series (site, year) of a date type
    out = {}
    for site in SITES:
        out.update({f"{site}_{k}": v for k, v in agreement(at_site(s, site), seen_at(site, target)).items()})
    return out


if SITES:
    site_single = pd.DataFrame([{"target": t, "candidate": label((c,)), **site_checks(t, s)}
                                for t, by_c in site_series.items() for c, s in by_c.items()]
                               + [{"target": t, "candidate": CHOICE, **site_checks(t, s)}
                                  for t, s in site_series_04.items()]).set_index(["target", "candidate"])
    for target in DATE_TYPES:
        print(f"{target} vs {PHENO_PARAMS[VALIDATE_WITH[target]]}" + (" (indirect)" if target == "start" else ""))
        display(site_single.xs(target, level="target").round(2))

    for site in SITES:
        fig, axes = plt.subplots(len(DATE_TYPES), len(METHODS), figsize=(12, 3.2 * len(DATE_TYPES)), sharex=True,
                                 squeeze=False)
        for i, target in enumerate(DATE_TYPES):
            seen = seen_at(site, target)
            for ax, method in zip(axes[i], METHODS):
                for name in INDICES:
                    s = at_site(site_series[target][(method, name)], site)
                    ax.plot(s.index, s.values, color=INDEX_COLORS[name], marker=INDEX_MARKERS[name], lw=1, ms=6,
                            label=name)
                ax.plot(seen.index, seen.values, lw=1.5, ms=5, label="observed", **REFERENCES["loc"][3])
                ax.set_title(f"{method}: {target} (observed: {PHENO_PARAMS[VALIDATE_WITH[target]]})", fontsize=10)
                ax.xaxis.get_major_locator().set_params(integer=True)
                ax.grid(alpha=0.3)
            axes[i, 0].set_ylabel("day of year")
        for ax in axes[-1]:
            ax.set_xlabel("year")
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=len(handles), fontsize=8)
        z = site_zones.loc[site]
        fig.suptitle(f"{site_names.get(site, site)}: AOI pixels within {SITE_RADIUS_M} m ({z.n_pixels:.0f} pixels, "
                     f"mean {z.elev_mean:.0f} m) vs the site's observations")
        plt.tight_layout(rect=(0, 0.04, 1, 1))
        plt.show()

## 6. Which version works best

- **Scored checks** (ranked 1 = best among the contenders of a date type): `r` (higher is better) and `rmse` (lower) against each validation source; for the peak and the end also the absolute `offset` against the local sites and the site areas (closer to 0 is better), as they lie at about the AOI's elevation. Not scored: the MeteoSwiss offset (the stations lie lower) and every offset of the start (the start comes before the observed colouring by definition).
- **`mean_rank`:** mean of a contender's ranks over the scored checks it has values for. Ties keep the table order (singles before pairs).
- **Reference row** `04 choice`: ranked against the contenders (its rank = 1 + the number of contenders better on that check) without changing their ranks. If it is itself one of the contenders (a pair of two top singles), both rows have the same values.
- **Plausibility** (`coverage`, `order_ok`, `days_per_100m`, `aoi_sd_years`) is shown but not scored: a contender with low coverage or many dates out of order should be read with care even if it ranks well.

**Heatmap:** the scored checks, one column each. Colour = rank (dark = best), the number in the cell = the value; * = correlation significant at 5 %. Mean rank in brackets.

Each correlation rests on 7–9 years, so differences of a few tenths in `r` between contenders are within the noise. A contender that does well across all sources is a sounder choice than one that wins a single check; restricting the pairs to the best singles (`BEST_TOP_N`) keeps the number of contenders, and with it the chance of a lucky winner, small.

### 6a. The single candidates

In [ ]:
SOURCE_NAMES = {"ms": "MeteoSwiss", "loc": "local sites", **{s: site_names.get(s, s) for s in SITES}}
PLAUSIBILITY = ["coverage", "order_ok", "days_per_100m", "aoi_sd_years"]


def checks_of(target, s, s_site=None):
    return {**aoi_checks(target, s), **(site_checks(target, s_site) if SITES else {})}


def scores(target, table):
    # (column, direction, label): "max" = higher is better, "min" = lower, "abs" = closer to 0
    out = []
    for key, source in SOURCE_NAMES.items():
        if f"{key}_r" not in table:
            continue
        out += [(f"{key}_r", "max", f"{source}\nr"), (f"{key}_rmse", "min", f"{source}\nRMSE (d)")]
        if key != "ms" and target != "start":
            out.append((f"{key}_offset", "abs", f"{source}\n|offset| (d)"))
    return out


def rank_table(target, table, references=(CHOICE,)):
    # Ranks of the contenders per scored check; reference rows ranked against them without changing their ranks
    sc = scores(target, table)

    def values(df):
        return pd.DataFrame({c: {"max": -df[c], "min": df[c], "abs": df[c].abs()}[d] for c, d, _ in sc}, index=df.index)

    refs = [r for r in references if r in table.index]
    v = values(table.drop(index=refs))
    ranks = v.rank(method="min")
    for name, row in values(table.loc[refs]).iterrows():
        ranks.loc[name] = [1 + (v[c] < row[c]).sum() if pd.notna(row[c]) else np.nan for c in v]
    out = table.assign(mean_rank=ranks.mean(axis=1)).sort_values("mean_rank", kind="stable")
    return out, ranks.loc[out.index], sc


def plot_ranks(title, ov, ranks, sc, references=(CHOICE,)):
    cmap = plt.get_cmap("Blues_r").copy()
    cmap.set_bad("#eeeeee")
    n = len(ov)
    fig, ax = plt.subplots(figsize=(1.3 * len(sc) + 4, 0.42 * n + 1.6))
    ax.imshow(np.ma.masked_invalid(ranks.values.astype(float)), cmap=cmap, vmin=1, vmax=n + 2, aspect="auto")
    for i, cand in enumerate(ov.index):
        for j, (col, direction, _) in enumerate(sc):
            v = ov.at[cand, col]
            if pd.isna(v):
                continue
            if direction == "max":
                text = f"{v:.2f}" + ("*" if ov.at[cand, col[:-2] + "_p"] < ALPHA else "")
            else:
                text = f"{v:.1f}" if direction == "min" else f"{v:+.1f}"
            ax.text(j, i, text, ha="center", va="center", fontsize=8,
                    color="white" if ranks.at[cand, col] <= n / 3 else "black")
    ax.set_xticks(range(len(sc)), [lbl for _, _, lbl in sc], fontsize=8)
    ax.set_yticks(range(n), [f"{cand}{' (reference)' if cand in references else ''}  ({ov.at[cand, 'mean_rank']:.1f})"
                             for cand in ov.index], fontsize=8)
    ax.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False, length=0)
    ax.set_xticks(np.arange(-0.5, len(sc)), minor=True)
    ax.set_yticks(np.arange(-0.5, n), minor=True)
    ax.grid(which="minor", color="white", lw=2)
    ax.tick_params(which="minor", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, fontsize=10, pad=40)
    plt.tight_layout()
    plt.show()


def validated_against(target):
    return PHENO_PARAMS[VALIDATE_WITH[target]] + (" (indirect)" if target == "start" else "")


single_table = single_checks.join(site_single) if SITES else single_checks
ranked_single, top_singles = {}, {}
for target in DATE_TYPES:
    ov, ranks, sc = rank_table(target, single_table.xs(target, level="target"))
    ranked_single[target] = ov
    top_singles[target] = [parse(c)[0] for c in ov.index if c != CHOICE][:BEST_TOP_N]
    print(f"{target.upper()} vs {validated_against(target)}: best {BEST_TOP_N}: "
          + "; ".join(label((c,)) for c in top_singles[target]))
    display(ov[["mean_rank", *[c for c, _, _ in sc], *PLAUSIBILITY]].round(2))
    plot_ranks(f"{target.capitalize()}, single candidates vs {validated_against(target)}: rank per check "
               "(dark = best)", ov, ranks, sc)

### 6b. Single candidates and pair averages

The pair averages (`combine`, section 4) of the `BEST_TOP_N` best singles of each date type join the 8 singles, and all are ranked again with the same checks, at the whole AOI and at the site areas. Below the heatmap, the scatter shows the whole-AOI anomaly against the observed anomaly per year for the best contenders and the reference (MeteoSwiss black squares, local sites purple diamonds); on the 1:1 line the anomalies agree.

In [ ]:
def contender_series(by_single, comps):
    return combine([by_single[c] for c in comps])


contenders = {t: [(c,) for c in CANDIDATES] + list(combinations(top_singles[t], 2)) for t in DATE_TYPES}
rows = []
for target, comps_list in contenders.items():
    for comps in comps_list:
        s_site = contender_series(site_series[target], comps) if SITES else None
        rows.append({"target": target, "candidate": label(comps), "n_parts": len(comps),
                     **checks_of(target, contender_series(series[target], comps), s_site)})
    rows.append({"target": target, "candidate": CHOICE, "n_parts": np.nan,
                 **checks_of(target, series_04[target], site_series_04.get(target))})
all_checks = pd.DataFrame(rows).set_index(["target", "candidate"])


def plot_scatter(target, ov, n_show=5):
    show = [c for c in ov.index if c != CHOICE][:n_show] + [CHOICE]
    ncols = 3
    nrows = int(np.ceil(len(show) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.5 * nrows), sharex=True, sharey=True, squeeze=False)
    for ax, cand in zip(axes.flat, show):
        est = aoi(series_04[target] if cand == CHOICE else contender_series(series[target], parse(cand))).dropna()
        for key in pheno:
            r = reference(key, target)
            common = est.index.intersection(r.index)
            a = est[common] - est[common].mean()
            s = r.anomaly[common] - r.anomaly[common].mean()
            style = REFERENCES[key][3]
            ax.scatter(s, a, marker=style["marker"], edgecolor=style["color"], s=30,
                       facecolor="white" if key == "loc" else style["color"], label=SOURCE_NAMES[key])
        ax.axline((0, 0), slope=1, color="grey", lw=0.8)
        ax.axhline(0, color="grey", lw=0.4)
        ax.axvline(0, color="grey", lw=0.4)
        r_text = ", ".join(f"{SOURCE_NAMES[k]} r = {ov.at[cand, f'{k}_r']:.2f}" for k in pheno
                           if pd.notna(ov.at[cand, f"{k}_r"]))
        title = cand.replace(" + ", " +\n") + (" (reference)" if cand == CHOICE else "")
        ax.set_title(f"{title}\n{r_text}", fontsize=8)
        ax.grid(alpha=0.3)
    for ax in axes.flat[len(show):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("observed anomaly (days)")
    for ax in axes[:, 0]:
        ax.set_ylabel(f"AOI {target} anomaly (days)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(handles), fontsize=8)
    fig.suptitle(f"{target.capitalize()}: whole-AOI anomaly vs {validated_against(target)} per year, "
                 f"best {n_show} contenders and the reference")
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()


ranked = {}
for target in DATE_TYPES:
    ov, ranks, sc = rank_table(target, all_checks.xs(target, level="target"))
    ranked[target] = ov
    print(f"{target.upper()} vs {validated_against(target)}: {len(ov) - 1} contenders "
          f"({len(CANDIDATES)} singles, {len(ov) - 1 - len(CANDIDATES)} pairs)")
    display(ov[["mean_rank", *[c for c, _, _ in sc], *PLAUSIBILITY]].round(2))
    plot_ranks(f"{target.capitalize()}, singles and pair averages vs {validated_against(target)}: rank per check "
               "(dark = best)", ov, ranks, sc)
    plot_scatter(target, ov)

### 6c. The choice per date type

`BEST_CHOICE` in `config.py` sets the version used from section 7 on, per date type:
- `"auto"`: the contender with the best mean rank in 6b (not the reference row).
- `"peak"` (start only): the start of the candidate(s) chosen for the peak, i.e. the same curve(s) and index(es). The start has no observation of its own and no candidate follows the needle colouring well (6a), so this is the default.
- a fixed single candidate or average, written as in the tables, e.g. `"Whittaker / GRVI"` or `"Whittaker / GRVI + double logistic / RCC"`.

The table compares each chosen version with `04 choice` on the main checks (`rank_in_6b` = its place in the ranking of 6b, if it is one of the contenders there).

In [ ]:
best_comps, how = {}, {}
for target in sorted(DATE_TYPES, key=lambda t: BEST_CHOICE[t] == "peak"):   # the peak first if the start follows it
    choice = BEST_CHOICE[target]
    if choice == "auto":
        best_comps[target] = parse([c for c in ranked[target].index if c != CHOICE][0])
        how[target] = "auto: best mean rank"
    elif choice == "peak":
        if target != "start" or BEST_CHOICE["peak"] == "peak":
            raise ValueError(f"BEST_CHOICE[{target!r}] = 'peak' is only possible for the start")
        best_comps[target] = best_comps["peak"]
        how[target] = "same curve(s) and index(es) as the peak"
    else:
        best_comps[target] = parse(choice)
        how[target] = "fixed in BEST_CHOICE"
best_comps = {t: best_comps[t] for t in DATE_TYPES}
choice_label = {t: label(c) for t, c in best_comps.items()}

MAIN = ["ms_r", "loc_r", *[f"{s}_r" for s in SITES], "ms_rmse", "loc_rmse", *[f"{s}_rmse" for s in SITES],
        "loc_offset", *[f"{s}_offset" for s in SITES], *PLAUSIBILITY]
rows = []
for target, comps in best_comps.items():
    s_site = contender_series(site_series[target], comps) if SITES else None
    order = list(ranked[target].drop(index=CHOICE).index)
    for name, checks in [(choice_label[target], checks_of(target, contender_series(series[target], comps), s_site)),
                         (CHOICE, checks_of(target, series_04[target], site_series_04.get(target)))]:
        place = order.index(name) + 1 if name in order else None
        rows.append({"target": target, "version": "chosen" if name != CHOICE else CHOICE, "candidate": name,
                     "how": how[target] if name != CHOICE else SEASON_CHOICE[target],
                     "rank_in_6b": f"{place} of {len(order)}" if place else "-",
                     **{k: v for k, v in checks.items() if k in MAIN}})
best_choice = pd.DataFrame(rows).set_index(["target", "version"])
display(best_choice.round(2))

## 7. The chosen dates

`selected` holds the chosen start / peak / end per zone and year (complete years), each from the version of 6c. The table gives the whole-AOI dates; the figure shows the season of the whole AOI per year with the reference data: needle colouring 50 % (squares) for the peak, needle drop 50 % (diamonds) for the end; black = MeteoSwiss stations, purple = local sites. The grey markers are `04 choice`.

In [ ]:
def doy_to_date(year, doy):
    return date(int(year), 1, 1) + timedelta(days=int(round(doy)) - 1) if pd.notna(doy) else None


for target in DATE_TYPES:
    print(f"{target}: {choice_label[target]} ({how[target]})")
selected = pd.DataFrame({t: contender_series(series[t], c) for t, c in best_comps.items()}).sort_index()
selected_04 = pd.DataFrame(series_04).sort_index()

z = selected.drop(index="all", level="zone").dropna()
print(f"start < peak < end in {((z.start < z.peak) & (z.peak < z.end)).mean():.0%} of {len(z)} zone-years")
aoi_sel = selected.xs("all", level="zone")
aoi_04 = selected_04.xs("all", level="zone")
display(aoi_sel.round(0).join(pd.DataFrame({f"{t}_date": [doy_to_date(y, v) for y, v in aoi_sel[t].items()]
                                            for t in DATE_TYPES}, index=aoi_sel.index)))

MONTH_TICKS = {date(2023, m, 1).timetuple().tm_yday: f"1 {date(2023, m, 1):%b}" for m in range(8, 13)}
fig, ax = plt.subplots(figsize=(11, 5))
ax.vlines(aoi_sel.index, aoi_sel.start, aoi_sel.end, color="tab:olive", lw=10, alpha=0.35, label="start to end")
ax.plot(aoi_sel.index, aoi_sel.start, "_", color="tab:olive", ms=18, mew=2, label=f"start ({choice_label['start']})")
ax.plot(aoi_sel.index, aoi_sel.peak, "o", color="darkorange", ms=9, label=f"peak ({choice_label['peak']})")
ax.plot(aoi_sel.index, aoi_sel.end, "_", color="saddlebrown", ms=18, mew=2, label=f"end ({choice_label['end']})")
for target, marker in [("start", "_"), ("peak", "o"), ("end", "_")]:
    ax.plot(aoi_04.index - 0.3, aoi_04[target], marker, color="grey", ms=7 if marker == "o" else 10, mew=1.5, ls="",
            label=CHOICE if target == "peak" else None)
for target, marker in [("peak", "s"), ("end", "D")]:
    for key in pheno:
        r = reference(key, target)
        ax.plot(r.index + (0.2 if key == "ms" else 0.35), r.doy, marker, color=REFERENCES[key][3]["color"], mfc="white",
                ms=6, ls="", label=f"{REFERENCES[key][1]} {PHENO_PARAMS[VALIDATE_WITH[target]]}")
ax.set_yticks(list(MONTH_TICKS), [f"{v} ({k})" for k, v in MONTH_TICKS.items()])
ax.set_xlabel("year")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3)
ax.set_title(f"Chosen season dates per year (whole AOI, mean {AOI_ELEV:.0f} m; tick labels: date (DOY))")
plt.tight_layout()
plt.show()

### 7b. The chosen dates at the local site

The chosen versions computed for the areas around the sites of `SITE_AREAS` (5b), against the site's own observations, with `04 choice` (grey) for comparison. The start is drawn with the observed needle colouring 50 % as an indirect reference.

In [ ]:
if SITES:
    for site in SITES:
        fig, axes = plt.subplots(1, len(DATE_TYPES), figsize=(5 * len(DATE_TYPES), 3.6), squeeze=False)
        for ax, target in zip(axes[0], DATE_TYPES):
            est = at_site(contender_series(site_series[target], best_comps[target]), site)
            ref = at_site(site_series_04[target], site)
            seen = seen_at(site, target)
            ax.plot(est.index, est.values, "o-", color="tab:blue", lw=1.5, ms=6, label="chosen")
            ax.plot(ref.index, ref.values, "o--", color="grey", lw=1, ms=5, label=CHOICE)
            ax.plot(seen.index, seen.values, lw=1.5, ms=5, label="observed", **REFERENCES["loc"][3])
            a = agreement(est, seen)
            stats_text = f"r = {a['r']:.2f}, offset {a['offset']:+.1f} d" if "r" in a else f"n = {a['n']}"
            ax.set_title(f"{target}: {choice_label[target]}\nvs {validated_against(target)}; {stats_text}", fontsize=9)
            ax.set_xlabel("year")
            ax.xaxis.get_major_locator().set_params(integer=True)
            ax.grid(alpha=0.3)
        axes[0, 0].set_ylabel("day of year")
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=len(handles), fontsize=8)
        fig.suptitle(f"{site_names.get(site, site)}: chosen dates of the AOI pixels within {SITE_RADIUS_M} m "
                     "vs the site's observations")
        plt.tight_layout(rect=(0, 0.07, 1, 1))
        plt.show()

## 8. Trend and short-term projection (no climate drivers)

For each chosen date (section 7) of the whole AOI and the complete years:
- **OLS trend** of the day of year on the year, with its 95 % confidence interval, extrapolated `FORECAST_YEARS` ahead with a 95 % prediction interval (the range a single future year is expected to fall in).
- **Mean model** (no trend) alongside. If the trend is not significant (`trend_significant` = False), the mean ± interval is the more honest expectation.

With about 9 years, one early or late year at either end moves the slope a lot.

In [ ]:
proj_rows = []
level = f"{1 - ALPHA:.0%}"
fig, axes = plt.subplots(1, len(DATE_TYPES), figsize=(5.5 * len(DATE_TYPES), 4.5), squeeze=False)
for ax, target in zip(axes[0], DATE_TYPES):
    d = aoi_sel[target].dropna()
    n = len(d)
    if n < 4:
        ax.set_title(f"{target}: only {n} years, skipped")
        continue
    years_fit = d.index.values.astype(float)
    y = d.values.astype(float)
    x0 = years_fit.mean()   # centred years for numerical stability
    ols = sm.OLS(y, sm.add_constant(years_fit - x0)).fit()
    mean_model = sm.OLS(y, np.ones((n, 1))).fit()
    future = np.arange(years_fit.max() + 1, years_fit.max() + 1 + FORECAST_YEARS)
    grid = np.arange(years_fit.min(), future.max() + 1)
    pred = ols.get_prediction(sm.add_constant(grid - x0, has_constant="add")).summary_frame(alpha=ALPHA)
    pred_mean = mean_model.get_prediction(np.ones((len(grid), 1))).summary_frame(alpha=ALPHA)
    slope_lo, slope_hi = ols.conf_int(ALPHA)[1]
    significant = bool(ols.pvalues[1] < ALPHA)
    print(f"{target} ({choice_label[target]}), {n} years: OLS slope {ols.params[1]:+.2f} d/yr "
          f"({level} CI {slope_lo:+.2f} to {slope_hi:+.2f}), p = {ols.pvalues[1]:.3f}, R2 = {ols.rsquared:.2f}")
    for i, yr in enumerate(grid):
        if yr in future:
            proj_rows.append({
                "date_type": target, "year": int(yr),
                "trend_doy": pred["mean"].iloc[i],
                "trend_pi_low": pred["obs_ci_lower"].iloc[i], "trend_pi_high": pred["obs_ci_upper"].iloc[i],
                "mean_doy": pred_mean["mean"].iloc[i],
                "mean_pi_low": pred_mean["obs_ci_lower"].iloc[i], "mean_pi_high": pred_mean["obs_ci_upper"].iloc[i],
                "trend_significant": significant,
            })

    color = "tab:blue"
    ax.fill_between(grid, pred["obs_ci_lower"], pred["obs_ci_upper"], color=color, alpha=0.15,
                    label=f"OLS {level} prediction interval")
    ax.fill_between(grid, pred["mean_ci_lower"], pred["mean_ci_upper"], color=color, alpha=0.3,
                    label=f"OLS {level} CI of the trend")
    ax.plot(grid, pred["mean"], color=color, lw=2, label="OLS trend")
    ax.plot(grid, pred_mean["mean"], "--", color="grey", lw=1, label="mean (no trend)")
    ax.fill_between(grid, pred_mean["obs_ci_lower"], pred_mean["obs_ci_upper"], facecolor="none",
                    edgecolor="grey", hatch="//", lw=0, alpha=0.5, label=f"mean {level} prediction interval")
    ax.plot(years_fit, y, "o", color="black", label="chosen date")
    ax.axvspan(future.min() - 0.5, future.max() + 0.5, color="grey", alpha=0.08)
    ax.set_title(f"{target} ({choice_label[target].replace(' + ', ' +' + chr(10))}): {ols.params[1]:+.2f} d/yr, "
                 f"p = {ols.pvalues[1]:.2f}", fontsize=9)
    ax.set_xlabel("year")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(handles), fontsize=8)
plt.suptitle("Trend and projection of the chosen dates (whole AOI; shaded columns = projected years)")
plt.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

season_projection = pd.DataFrame(proj_rows)
if not season_projection.empty:
    season_projection = season_projection.set_index(["date_type", "year"]).round(1)
    for c in ["trend_doy", "mean_doy"]:
        season_projection[c.replace("_doy", "_date")] = [doy_to_date(yr, v) for (_, yr), v in season_projection[c].items()]
    display(season_projection)

## 9. Phenology vs topography (elevation band × aspect)

Uses the chosen dates of section 7 per zone and year.

- **Map:** elevation band and aspect class of the AOI pixels (`zones.tif` from `01`) on a hillshade. The hillshade is lit from the south, so north-facing slopes appear dark: blue (N) pixels should sit on the dark side of the valleys.
- **Table:** zone attributes, the mean number of clear autumn dates per year and index (`autumn_dates_*`, `AUTUMN_START_DOY`–`POST_WINDOW[1]`) and the mean share of SCL "dark" pixels in autumn.
- **Plots:** day of year against elevation band, one line per aspect (mean ± sd over years), first for all date types, then for the peak alone with the local sites only. The validation stations and sites (MeteoSwiss black squares, local sites purple diamonds) are drawn at their own height with their mean ± sd over the complete years: needle colouring 50 % for the peak, needle drop 50 % for the end.
- **Model:** OLS `doy ~ elev_100m + C(aspect) + C(year)`.
  - The elevation coefficient is in days per 100 m.
  - The aspect coefficient is the north − south difference in days.
  - `C(year)` absorbs year-to-year weather.

**Caveats:**
- Zones share weather and clouds within a year, so the CIs are somewhat optimistic.
- North slopes lose more autumn dates to SCL class 2 (dark / topographic shadow); see `autumn_dates_*` and `autumn_dark_fraction`.

In [ ]:
zr = rioxarray.open_rasterio(CACHE_DIR / "zones.tif", masked=True)
elev, elev_band, aspect = (zr.sel(band=b).values for b in (1, 2, 3))
on = ~np.isnan(elev_band)
yy, xx = np.meshgrid(zr.y.values, zr.x.values, indexing="ij")
left, bottom, right, top = zr.rio.bounds()
shade = LightSource(azdeg=180, altdeg=30).hillshade(np.nan_to_num(elev, nan=np.nanmean(elev)),
                                                   dx=RESOLUTION, dy=RESOLUTION)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax in axes:
    ax.imshow(shade, cmap="gray", extent=(left, right, bottom, top), alpha=0.6)
bands = np.unique(elev_band[on])
for b, color in zip(bands, plt.get_cmap("viridis")(np.linspace(0, 1, len(bands)))):
    sel = on & (elev_band == b)
    axes[0].scatter(xx[sel], yy[sel], s=0.2, color=color, label=f"{int(b)}-{int(b + ELEV_STEP)} m")
for asp, code_ in ASPECT_CODES.items():
    sel = on & (aspect == code_)
    axes[1].scatter(xx[sel], yy[sel], s=0.2, color=ASPECT_COLORS[asp], label=f"{asp} ({int(sel.sum())} px)")
for ax, title in zip(axes, ["Elevation band of the AOI pixels", "Aspect class (hillshade lit from the south)"]):
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(markerscale=20, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
zone_attrs = zone_table.drop(index="all")[["elev_band", "aspect", "elev_mean"]]
autumn = stats_all[stats_all.index.dayofyear >= AUTUMN_START_DOY]
autumn_dates = pd.DataFrame({
    f"autumn_dates_{name}": {zone: o[(o.w > 0) & o.doy.between(AUTUMN_START_DOY, POST_WINDOW[1])]
                             .groupby("year").size().mean() for zone, o in obs_by_zone[name].items()}
    for name in INDICES})
coverage = (zone_table[["elev_band", "aspect", "n_pixels", "elev_mean", "slope_mean", "northness_mean"]]
            .join(autumn_dates)
            .join(autumn.groupby("zone").scl_dark_fraction.mean().rename("autumn_dark_fraction")))
display(coverage.round(2))

zone_dates = selected.drop(index="all", level="zone").reset_index().join(zone_attrs, on="zone", how="inner")
TOPO_REFERENCE = {"peak": "mlard94d", "end": "mlard95d"}   # observed phase drawn per date type


def station_dates(target, sources=REFERENCES):
    # Validation data of one date type per station / site: [(source key, mean and sd of the DOY over the complete
    # years, name, height)]; sources: keys of REFERENCES
    if target not in TOPO_REFERENCE:
        return []
    out = []
    for key, (file, _, _, _) in REFERENCES.items():
        if key not in sources or not (VALIDATION_DIR / file).exists():
            continue
        p = pd.read_csv(VALIDATION_DIR / file)
        p = p[(p.param == TOPO_REFERENCE[target]) & p.year.isin(years)]
        out.append((key, p.groupby("station").agg(name=("name", "first"), height=("height_masl", "first"),
                                                  mean=("doy", "mean"), std=("doy", "std"))))
    return out


def plot_topography(targets, names=False, sources=REFERENCES, title=None, zone_names=None):
    # Chosen dates per elevation band and aspect, with the validation stations / sites at their own height;
    # names: label each station / site (above its whisker); sources: keys of REFERENCES to draw;
    # title: replaces the figure and panel titles; zone_names: legend entry of the zone lines per aspect
    width = 12 if len(targets) == 1 else 5.5 * len(targets) + 1
    fig, axes = plt.subplots(1, len(targets), figsize=(width, 5.5), sharex=True, squeeze=False, layout="constrained")
    for ax, target in zip(axes[0], targets):
        for asp, color in ASPECT_COLORS.items():
            d = zone_dates[zone_dates.aspect == asp].dropna(subset=[target])
            if d.empty:
                continue
            shift = ELEV_STEP / 2 + {"N": -0.08, "S": 0.08, "flat": 0}[asp] * ELEV_STEP   # band centre, dodged
            ax.scatter(d.elev_band + shift, d[target], s=8, color=color, alpha=0.3)
            g = d.groupby("elev_band")[target].agg(["mean", "std"])
            ax.errorbar(g.index + shift, g["mean"], yerr=g["std"].fillna(0), fmt="o-", color=color, capsize=3,
                        label=(zone_names or {}).get(asp, f"AOI zones, aspect {asp}"))
        tags = []
        for key, st in station_dates(target, sources):
            _, source, kind, style = REFERENCES[key]
            ax.errorbar(st.height, st["mean"], yerr=st["std"].fillna(0), fmt=style["marker"], color=style["color"],
                        mfc="white", ms=7, capsize=3, label=f"{source} {PHENO_PARAMS[TOPO_REFERENCE[target]]} ({kind})")
            tags += [(r.height, r["mean"] + np.nan_to_num(r["std"]), r["name"], style["color"]) for _, r in st.iterrows()]
        if names:
            last, dx = -np.inf, 0
            for height, y, name, color in sorted(tags):
                dx = dx + 9 if height - last < 25 else 0   # names of stations closer than 25 m side by side
                last = height
                ax.annotate(name, (height, y), xytext=(dx, 4), textcoords="offset points", rotation=90, ha="center",
                            va="bottom", fontsize=7, color=color)
            lo, hi = ax.get_ylim()
            ax.set_ylim(lo, hi + 0.2 * (hi - lo))   # room for the names
        if title is None:
            ax.set_title(f"{target} ({choice_label[target].replace(' + ', ' +' + chr(10))})", fontsize=9)
        ax.set_xlabel("elevation (m)")
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel("day of year")
    labels = {}
    for ax in axes[0]:
        for h, lbl in zip(*ax.get_legend_handles_labels()):
            labels.setdefault(lbl, h)
    fig.legend(labels.values(), labels.keys(), loc="outside lower center", ncol=3, fontsize=8)
    kinds = " / ".join(REFERENCES[k][2] for k in sources)
    fig.suptitle(title or f"Chosen dates vs elevation and aspect, with the validation {kinds} at their height "
                          "(mean ± sd over years)", fontsize=11)
    plt.show()


plot_topography(list(DATE_TYPES))
plot_topography(["peak"], names=True, sources=["loc"], title="Peak of Yellowing per elevation and aspect",
                zone_names={"N": "Larch Zones North", "S": "Larch Zones South", "flat": "Larch Zones Flat"})

NS_TERM = "C(aspect, Treatment('S'))[T.N]"


def topo_model(data, col):
    d = data.dropna(subset=[col]).assign(elev_100m=lambda x: x.elev_mean / 100)
    if (not {"N", "S"} <= set(d.aspect) or d.elev_band.nunique() < 2
            or len(d) < d.year.nunique() + d.aspect.nunique() + 3):
        return None
    return smf.ols(f"{col} ~ elev_100m + C(aspect, Treatment('S')) + C(year)", data=d).fit()


rows = []
for target in DATE_TYPES:
    fit = topo_model(zone_dates, target)
    if fit is None:
        print(f"{target}: too few zone-years (needs N and S zones and >= 2 elevation bands), no model")
        continue
    ci = fit.conf_int(ALPHA)
    rows.append({
        "date_type": target, "method": choice_label[target], "n_zone_years": int(fit.nobs),
        "n_zones": zone_dates.dropna(subset=[target]).zone.nunique(),
        "days_per_100m": fit.params["elev_100m"],
        "elev_ci_low": ci.loc["elev_100m", 0], "elev_ci_high": ci.loc["elev_100m", 1],
        "elev_p": fit.pvalues["elev_100m"],
        "north_minus_south_days": fit.params[NS_TERM],
        "ns_ci_low": ci.loc[NS_TERM, 0], "ns_ci_high": ci.loc[NS_TERM, 1], "ns_p": fit.pvalues[NS_TERM],
        "r2": fit.rsquared,
    })
topo_effects = pd.DataFrame(rows).set_index("date_type").round(3) if rows else pd.DataFrame()
display(topo_effects)

## 10. Trend / seasonal / remainder decomposition (STL, whole AOI)

Robust STL on a regular 5-day grid splits each index into a long-term trend, a repeating yearly cycle and the remainder. It needs at least two full years.

In [ ]:
STEP_DAYS = 5
PERIOD = int(round(365.25 / STEP_DAYS))   # observations per year on the regular grid


def stl_decompose(series_, name):
    s = series_.resample(f"{STEP_DAYS}D").mean()
    s = s.loc[s.first_valid_index():s.last_valid_index()].interpolate(method="time")
    if len(s) < 2 * PERIOD:
        print(f"{name}: {len(s) * STEP_DAYS / 365.25:.1f} years of data; STL needs at least 2 full years")
        return None
    res = STL(s, period=PERIOD, robust=True).fit()
    return pd.DataFrame({"observed": s, "trend": res.trend, "seasonal": res.seasonal, "remainder": res.resid})


for name in INDICES:
    dec = stl_decompose(medians[name], name)
    if dec is None:
        continue
    color = INDEX_COLORS[name]
    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    axes[0].plot(dec.index, dec.observed, color="grey", lw=1)
    axes[0].set_ylabel(f"{name} observed")
    axes[1].plot(dec.index, dec.trend, color=color, lw=2)
    axes[1].set_ylabel("trend")
    axes[2].plot(dec.index, dec.seasonal, color=color, lw=1.5)
    axes[2].set_ylabel("seasonal")
    axes[3].scatter(dec.index, dec.remainder, s=6, color=color)
    axes[3].set_ylabel("remainder")
    for a in axes[2:]:
        a.axhline(0, color="black", lw=0.5)
    for a in axes:
        a.grid(alpha=0.3)
    axes[0].set_title(f"{name} STL decomposition (period = {PERIOD} x {STEP_DAYS} days)")
    plt.tight_layout()
    plt.show()
    slope = np.polyfit((dec.index - dec.index[0]).days / 365.25, dec.trend.values, 1)[0]
    print(f"{name} trend slope: {slope:+.4f} per year | seasonal amplitude: "
          f"{dec.seasonal.max() - dec.seasonal.min():.3f} | remainder std: {dec.remainder.std():.3f}")

## 11. Export

Only files of this notebook are written; those of `04` (`season_selected.csv` etc., read by `05_climate_analysis`) are left as they are.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)


def long_table(by_target, by_target_04, unit):
    # Dates of every contender of 6b and the reference per date type: target, candidate, <unit>, year, doy, date
    parts = {}
    for t, by_single in by_target.items():
        for comps in contenders[t]:
            parts[(t, label(comps))] = contender_series(by_single, comps)
        parts[(t, CHOICE)] = by_target_04[t]
    out = pd.concat(parts, names=["target", "candidate"]).rename("doy").reset_index()
    out["date"] = pd.to_datetime(out.year.astype(str)) + pd.to_timedelta(out.doy.round() - 1, unit="D")
    return out


long_table(series, series_04, "zone").to_csv(OUT_DIR / "season_all_indices_dates.csv", index=False)
if SITES:
    long_table(site_series, site_series_04, "site").to_csv(OUT_DIR / "season_all_indices_site_dates.csv", index=False)
comparison_out = pd.concat({"single": pd.concat(ranked_single, names=["target"]),
                            "with pairs": pd.concat(ranked, names=["target"])}, names=["ranking"])
comparison_out.to_csv(OUT_DIR / "season_all_indices_comparison.csv")
best_choice.to_csv(OUT_DIR / "season_best_choice.csv")
selected.to_csv(OUT_DIR / "season_selected_best.csv")
season_projection.to_csv(OUT_DIR / "season_projection_best.csv")
topo_effects.to_csv(OUT_DIR / "topography_effects_best.csv")
print(f"Results written to {OUT_DIR}")

## References

- Atzberger, C. & Eilers, P. H. C. (2011). A time series for monitoring vegetation activity and phenology at 10-daily time steps covering large parts of South America. *International Journal of Digital Earth* 4(5), 365–386.
- Beck, P. S. A., Atzberger, C., Høgda, K. A., Johansen, B. & Skidmore, A. K. (2006). Improved monitoring of vegetation dynamics at very high latitudes: a new method using MODIS NDVI. *Remote Sensing of Environment* 100(3), 321–334.
- Copernicus Land Monitoring Service (2021). *HR-VPP Algorithm Theoretical Basis Document: Seasonal Trajectories and VPP parameters*, issue 1.1. https://land.copernicus.eu/en/technical-library
- Eilers, P. H. C. (2003). A perfect smoother. *Analytical Chemistry* 75(14), 3631–3636.
- Gray, J., Sulla-Menashe, D. & Friedl, M. A. (2019). *User Guide to Collection 6 MODIS Land Cover Dynamics (MCD12Q2) Product*. NASA LP DAAC.
- Jönsson, P. & Eklundh, L. (2004). TIMESAT – a program for analyzing time-series of satellite sensor data. *Computers & Geosciences* 30(8), 833–845.
- MeteoSwiss. Phenological observations, open data (`ch.meteoschweiz.ogd-phenology`). https://opendatadocs.meteoswiss.ch/a-data-groundbased/a9-phenological-observations
- Local larch phenology observations (`Lärchen_pheno.xlsx`): sites Il Fuorn, Val Cluozza, Val Mingèr, Val Trupchun and Lü.
- Merzlyak, M. N., Gitelson, A. A., Chivkunova, O. B. & Rakitin, V. Y. (1999). Non-destructive optical detection of pigment changes during leaf senescence and fruit ripening. *Physiologia Plantarum* 106(1), 135–141.
- Migliavacca, M., Cremonese, E., Colombo, R., et al. (2008). European larch phenology in the Alps: can we grasp the role of ecological factors by combining field observations and inverse modelling? *International Journal of Biometeorology* 52(7), 587–605.
- Motohka, T., Nasahara, K. N., Oguma, H. & Tsuchida, S. (2010). Applicability of green-red vegetation index for remote sensing of vegetation phenology. *Remote Sensing* 2(10), 2369–2387.
- Richardson, A. D., Hufkens, K., Milliman, T., et al. (2018). Tracking vegetation phenology across diverse North American biomes using PhenoCam imagery. *Scientific Data* 5, 180028.
- Sonnentag, O., Hufkens, K., Teshera-Sterne, C., et al. (2012). Digital repeat photography for phenological research in forest ecosystems. *Agricultural and Forest Meteorology* 152, 159–177.
- Tian, F., Cai, Z., Jin, H., et al. (2021). Calibrating vegetation phenology from Sentinel-2 using eddy covariance, PhenoCam, and PEP725 networks across Europe. *Remote Sensing of Environment* 260, 112456.
- Zeng, L., Wardlow, B. D., Xiang, D., Hu, S. & Li, D. (2020). A review of vegetation phenological metrics extraction using time-series, multispectral satellite data. *Remote Sensing of Environment* 237, 111511.